In [1]:
import pandas as pd
import plotly.express as px

In [3]:
df_apps = pd.read_csv('data/apps.csv')

# Drop columns that are completely unnecessary for our visual charting targets
df_apps.drop(['Last_Updated', 'Android_Ver'], axis=1, inplace=True)

# Purge any rows containing blank ratings
df_apps_clean = df_apps.dropna()

In [4]:
df_apps_clean = df_apps_clean.drop_duplicates(subset=['App', 'Type', 'Price'])

In [5]:
# Strip commas from Installs column entries
df_apps_clean.Installs = df_apps_clean.Installs.astype(str).str.replace(',', "")
df_apps_clean.Installs = pd.to_numeric(df_apps_clean.Installs)

# Strip dollar signs from Price column entries
df_apps_clean.Price = df_apps_clean.Price.astype(str).str.replace('$', "")
df_apps_clean.Price = pd.to_numeric(df_apps_clean.Price)

In [6]:
# Keep only legitimate marketplace apps
df_apps_clean = df_apps_clean[df_apps_clean['Price'] < 250]

# Calculate total estimated gross revenue returns
df_apps_clean['Revenue_Estimate'] = df_apps_clean.Installs.mul(df_apps_clean.Price)

In [7]:
ratings = df_apps_clean.Content_Rating.value_counts()

fig = px.pie(labels=ratings.index, values=ratings.values, title="Content Rating Demographics", names=ratings.index, hole=0.6)
fig.update_traces(textposition='inside', textfont_size=15, textinfo='percent')
fig.show()

In [8]:
cat_number = df_apps_clean.groupby('Category').agg({'App': pd.Series.count})
category_installs = df_apps_clean.groupby('Category').agg({'Installs': pd.Series.sum})

cat_merged_df = pd.merge(cat_number, category_installs, on='Category', how="inner")

fig = px.scatter(cat_merged_df, x='App', y='Installs', title='Category Concentration', size='App', hover_name=cat_merged_df.index, color='Installs')
fig.update_layout(xaxis_title="Number of Apps", yaxis_title="Installs", yaxis=dict(type='log'))
fig.show()

In [9]:
stack = df_apps_clean.Genres.str.split(';', expand=True).stack()
num_genres = stack.value_counts()

# Plot the leaderboard using a custom continuous color scale gradient
fig = px.bar(x=num_genres.index[:15], y=num_genres.values[:15], title='Top Genres', color=num_genres.values[:15], color_continuous_scale='Agsunset')
fig.update_layout(xaxis_title='Genre', yaxis_title='Number of Apps', coloraxis_showscale=False)
fig.show()

In [10]:
# Check download metrics lost by adding a price tag
box = px.box(df_apps_clean, y='Installs', x='Type', color='Type', notched=True, points='all', title='Installs: Free vs Paid')
box.update_layout(yaxis=dict(type='log'))
box.show()

# Isolate paid software to spot optimal monetization price points
df_paid_apps = df_apps_clean[df_apps_clean['Type'] == 'Paid']
price_box = px.box(df_paid_apps, x='Category', y='Price', title='Price Benchmarks per Category')
price_box.update_layout(xaxis={'categoryorder':'max descending'}, yaxis=dict(type='log'))
price_box.show()

In [11]:
# 1. Print the clean structural row and column dimensions
print(df_apps_clean.shape)

# 2. Extract a completely random sample of 5 rows to see what the fields look like
df_apps_clean.sample(5)

(8184, 11)


,App,Category,Rating,Reviews,Size_MBs,Installs,Type,Price,Content_Rating,Genres,Revenue_Estimate
1721,Lottery Ticket Checker - Florida Results & Lotto,TOOLS,1.0,3,41.0,500,Free,0.0,Everyone,Tools,0.0
8847,Texas Holdem & Omaha Poker: Pokerist,GAME,4.3,187200,28.0,10000000,Free,0.0,Teen,Card,0.0
8925,Phonto - Text on Photos,PHOTOGRAPHY,4.3,307453,17.0,10000000,Free,0.0,Everyone,Photography,0.0
3599,CV Maker Pro,BUSINESS,3.7,113,8.6,10000,Free,0.0,Everyone,Business,0.0
6779,HomeWork,EDUCATION,4.3,16195,5.2,1000000,Free,0.0,Everyone,Education,0.0


In [12]:
# Strip duplicates using an explicit subset definition constraint
df_apps_clean = df_apps_clean.drop_duplicates(subset=['App', 'Type', 'Price'])

# Print our new milestone row target count dimension size
print(df_apps_clean.shape)

(8184, 11)


In [13]:
# 1. Aggregate counts for free versus paid apps per category sector
df_free_vs_paid = df_apps_clean.groupby(["Category", "Type"], as_index=False).agg({'App': pd.Series.count})

# 2. Build the interactive grouped vertical bar chart layout
g_bar = px.bar(
    df_free_vs_paid,
    x='Category',
    y='App',
    title='Free vs Paid Apps by Category',
    color='Type',
    barmode='group'
)

# 3. Fine-tune axes limits and sorting order properties
g_bar.update_layout(
    xaxis_title='Category',
    yaxis_title='Number of Apps',
    xaxis={'categoryorder':'total descending'},
    yaxis=dict(type='log') # Keeps paid bars visible alongside massive free counts
)
g_bar.show()